In [ ]:
# NB : This reload your library after each edit, 
# so you don't have to restart the kernel
%load_ext autoreload
%autoreload 2

In [15]:
# --- Imports (replace the TF ones) ---
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from herbs_detection.model import predict, predict_top3, predict_set

In [ ]:
PROJECT_ROOT = Path("../")
ALL_IMAGES_DIR = PROJECT_ROOT / "data/raw/all_images"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FROZEN = 10
EPOCHS_FINE = 20

In [ ]:


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# --- Load labels ---
labels = pd.read_csv(PROJECT_ROOT / "data/files_df.csv")
# match images in all_images/ by basename of the original path
labels["basename"] = labels["filename"].apply(lambda f: Path(f).name)
print(labels["name"].value_counts())
print(f"\nTotal: {len(labels)} images")

In [ ]:
# Build a dataframe from filenames in all_images/
file_names = sorted([p.name for p in ALL_IMAGES_DIR.iterdir() if p.is_file()])

files_df = pd.DataFrame({"filename": file_names})
files_df["name"] = files_df["filename"].str.split("_", n=1).str[0]

print(f"Found {len(files_df)} files in {ALL_IMAGES_DIR}")
files_df.head()

In [ ]:
# --- Load images from all_images/ as numpy arrays ---
X, y = [], []
missing = []

for _, row in labels.iterrows():
    img_path = ALL_IMAGES_DIR / row["basename"]
    if not img_path.exists():
        missing.append(row["basename"])
        continue
    img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    X.append(np.array(img))
    y.append(row["name"])

if missing:
    print(f"Warning: {len(missing)} images not found in all_images/")

X = np.array(X, dtype=np.float32) / 255.0   # shape: (N, 224, 224, 3), normalized to [0, 1]
y = np.array(y)

le = LabelEncoder()
y_enc = le.fit_transform(y)                  # integer labels
NUM_CLASSES = len(le.classes_)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y_enc.shape}")
print(f"Classes ({NUM_CLASSES}): {le.classes_}")



In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

class PlantDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X          # (N, 224, 224, 3) float32 in [0,1]
        self.y = y
        self.augment = augment
        self.normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                              [0.229, 0.224, 0.225])
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]) if augment else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.X[idx]).permute(2, 0, 1)  # (3, H, W)
        if self.aug:
            img = self.aug(img)
        img = self.normalize(img)
        return img, int(self.y[idx])


In [ ]:
# --- Dataset ---

from sklearn.model_selection import train_test_split

idx = np.arange(len(X))
idx_train, idx_test = train_test_split(idx, test_size=0.15, stratify=y_enc, random_state=42)
idx_train, idx_val  = train_test_split(idx_train, test_size=0.15, stratify=y_enc[idx_train], random_state=42)

train_loader = DataLoader(PlantDataset(X[idx_train], y_enc[idx_train], augment=True),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(PlantDataset(X[idx_val],   y_enc[idx_val]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(PlantDataset(X[idx_test],  y_enc[idx_test]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)



In [ ]:
# --- Model: ResNet18 pretrained ---
model = models.resnet18(weights="IMAGENET1K_V1")

# Phase 1: freeze backbone, replace head
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # only head is trainable

model = model.to(DEVICE)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct = 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labs in loader:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labs)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (out.argmax(1) == labs).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# --- Phase 1: head only ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

print("Phase 1 — frozen backbone")
for epoch in range(EPOCHS_FROZEN):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch+1:02d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")


In [ ]:
# --- Phase 2: unfreeze all layers, fine-tune with low LR ---
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val, patience_cnt, best_state = 0, 0, None
print("\nPhase 2 — fine-tuning")
for epoch in range(EPOCHS_FINE):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step(vl)
    print(f"Epoch {epoch+1:02d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
    if va > best_val:
        best_val = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}; patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= 5:
            print("Early stopping"); break

model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val:.3f}")


In [ ]:
# --- Evaluation ---
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labs in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(labs.tolist())

print(classification_report(y_true, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("ResNet18 — Confusion Matrix")
plt.tight_layout(); plt.show()


## K-Fold Cross-Validation Benchmarking

`StratifiedKFold` is applied to the in-memory image array `X` — no extra disk I/O per fold.
A fresh ResNet18 is trained from scratch for each fold using the same two-phase strategy
(frozen backbone → fine-tune), and the best validation accuracy per fold is recorded.

Tune `KF_EPOCHS_FROZEN` / `KF_EPOCHS_FINE` to trade accuracy for speed:
- Fewer epochs = faster benchmark, slightly noisier estimates.
- Match the main training section's epoch counts for the most faithful comparison.

In [ ]:
from sklearn.model_selection import StratifiedKFold

KF_SPLITS        = 5
KF_EPOCHS_FROZEN = 5    # head-only warmup (keep short to save time)
KF_EPOCHS_FINE   = 10   # fine-tuning phase
KF_PATIENCE      = 3    # early-stopping patience per fold

skf_pt = StratifiedKFold(n_splits=KF_SPLITS, shuffle=True, random_state=42)
kf_fold_accs = []


def _build_resnet(num_classes: int, device):
    """Fresh ResNet18 with frozen backbone and a new head."""
    m = models.resnet18(weights="IMAGENET1K_V1")
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)


def _run_fold_epoch(model, loader, optimizer=None):
    """One training or evaluation epoch. Pass optimizer=None for eval."""
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    _crit = nn.CrossEntropyLoss()
    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(train_mode):
        for imgs, labs in loader:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            out  = model(imgs)
            loss = _crit(out, labs)
            if train_mode:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (out.argmax(1) == labs).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


for fold, (tr_idx, vl_idx) in enumerate(skf_pt.split(X, y_enc)):
    print(f"\n── Fold {fold + 1}/{KF_SPLITS} ──────────────────────────────────────")

    tr_loader = DataLoader(
        PlantDataset(X[tr_idx], y_enc[tr_idx], augment=True),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    )
    vl_loader = DataLoader(
        PlantDataset(X[vl_idx], y_enc[vl_idx]),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    )

    fold_model = _build_resnet(NUM_CLASSES, DEVICE)

    # ── Phase 1: head only ────────────────────────────────────────────────
    opt1 = torch.optim.Adam(fold_model.fc.parameters(), lr=1e-3)
    for ep in range(KF_EPOCHS_FROZEN):
        tl, ta = _run_fold_epoch(fold_model, tr_loader, optimizer=opt1)
        vl, va = _run_fold_epoch(fold_model, vl_loader)
        print(f"  [frozen]   ep {ep + 1:02d} | train acc {ta:.3f} | val acc {va:.3f}")

    # ── Phase 2: unfreeze + fine-tune ────────────────────────────────────
    for p in fold_model.parameters():
        p.requires_grad = True

    opt2  = torch.optim.Adam(fold_model.parameters(), lr=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, patience=2, factor=0.5)

    best_va, patience_cnt = 0.0, 0
    for ep in range(KF_EPOCHS_FINE):
        tl, ta = _run_fold_epoch(fold_model, tr_loader, optimizer=opt2)
        vl, va = _run_fold_epoch(fold_model, vl_loader)
        sched.step(vl)
        print(f"  [finetune] ep {ep + 1:02d} | train acc {ta:.3f} | val acc {va:.3f}")
        if va > best_va:
            best_va = va
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= KF_PATIENCE:
                print("  Early stopping")
                break

    print(f"  → Best val acc fold {fold + 1}: {best_va:.4f}")
    kf_fold_accs.append(best_va)
    del fold_model          # free GPU memory
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n── K-Fold Summary ({KF_SPLITS} folds) ──────────────────────────────────")
for i, acc in enumerate(kf_fold_accs):
    print(f"  Fold {i + 1}: {acc:.4f}")
print(f"  Mean : {np.mean(kf_fold_accs):.4f}")
print(f"  Std  : {np.std(kf_fold_accs):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

fold_labels = [f"Fold {i + 1}" for i in range(KF_SPLITS)]
mean_acc = np.mean(kf_fold_accs)
std_acc  = np.std(kf_fold_accs)

bar_colors = ["#4C72B0" if a >= mean_acc else "#DD8452" for a in kf_fold_accs]
bars = ax.bar(fold_labels, kf_fold_accs, color=bar_colors, alpha=0.85)

ax.axhline(mean_acc, color="red", linestyle="--", linewidth=1.5,
           label=f"Mean = {mean_acc:.4f}")
ax.fill_between(
    [-0.5, KF_SPLITS - 0.5],
    mean_acc - std_acc, mean_acc + std_acc,
    alpha=0.12, color="red", label=f"±1 std = {std_acc:.4f}",
)

ax.set_xlim(-0.5, KF_SPLITS - 0.5)
ax.set_ylabel("Val Accuracy")
ax.set_ylim(0, 1)
ax.set_title(f"ResNet18 K-Fold CV (k={KF_SPLITS})")
ax.bar_label(bars, labels=[f"{a:.3f}" for a in kf_fold_accs], padding=4, fontsize=9)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
save_dir = PROJECT_ROOT / "backend/app/models"
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
from sklearn.preprocessing import LabelEncoder
import pickle


# Save model weights (equivalent to model.save_weights() in Keras)
torch.save(model.state_dict(), save_dir / "resnet18_plants_2.pt")

# Save the LabelEncoder too — you need it to decode predictions back to species names
with open(save_dir / "label_encoder_2.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved.")


In [ ]:
# ---------------------------------------------------------------------------
# Deploy model files to Google Cloud Storage
#   - If models already exist in the bucket, archive them with their creation
#     date as suffix before uploading the new ones.
# ---------------------------------------------------------------------------
import os
from datetime import timezone
from pathlib import Path
from google.cloud import storage

GCS_BUCKET_NAME   = os.environ.get("GCS_BUCKET_NAME", "plant-detect-models")
GCS_MODELS_PREFIX = os.environ.get("GCS_MODELS_PREFIX", "models").rstrip("/")
GCS_ARCHIVE_PREFIX = f"{GCS_MODELS_PREFIX}/archive"

_files_to_upload = {
    "resnet18_plants.pt": save_dir / "resnet18_plants.pt",
    "label_encoder.pkl":  save_dir / "label_encoder.pkl",
}

client = storage.Client()
bucket = client.bucket(GCS_BUCKET_NAME)

# ── 1. Archive existing models if present ────────────────────────────────
for filename in _files_to_upload:
    src_blob_name = f"{GCS_MODELS_PREFIX}/{filename}"
    src_blob = bucket.blob(src_blob_name)

    if src_blob.exists():
        # Use blob creation time as archive suffix
        src_blob.reload()
        created_at = src_blob.time_created.astimezone(timezone.utc)
        date_suffix = created_at.strftime("%Y%m%d_%H%M%S")

        stem, ext = filename.rsplit(".", 1)
        archive_name = f"{GCS_ARCHIVE_PREFIX}/{stem}_{date_suffix}.{ext}"

        bucket.copy_blob(src_blob, bucket, archive_name)
        print(f"Archived gs://{GCS_BUCKET_NAME}/{src_blob_name} → {archive_name}")

# ── 2. Upload new model files ─────────────────────────────────────────────
for filename, local_path in _files_to_upload.items():
    blob_name = f"{GCS_MODELS_PREFIX}/{filename}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(local_path))
    print(f"Uploaded  {local_path.name} → gs://{GCS_BUCKET_NAME}/{blob_name}")

print("\nDeploy complete.")
print(f"Archives stored under gs://{GCS_BUCKET_NAME}/{GCS_ARCHIVE_PREFIX}/")

In [ ]:
# ---------------------------------------------------------------------------
# Download model files from GCS → backend/app/models/gcp_model/
# ---------------------------------------------------------------------------
import os
from pathlib import Path
from google.cloud import storage

GCS_BUCKET_NAME   = os.environ.get("GCS_BUCKET_NAME", "plant-detect-models")
GCS_MODELS_PREFIX = os.environ.get("GCS_MODELS_PREFIX", "models").rstrip("/")

dest_dir = PROJECT_ROOT / "backend/app/models/gcp_model"
dest_dir.mkdir(parents=True, exist_ok=True)

_files_to_download = [
    "resnet18_plants.pt",
    "label_encoder.pkl",
]

client = storage.Client()
bucket = client.bucket(GCS_BUCKET_NAME)

for filename in _files_to_download:
    blob_name = f"{GCS_MODELS_PREFIX}/{filename}"
    dest_path = dest_dir / filename
    bucket.blob(blob_name).download_to_filename(str(dest_path))
    print(f"Downloaded gs://{GCS_BUCKET_NAME}/{blob_name} → {dest_path}")

print("\nAll model files downloaded from GCS successfully.")


In [ ]:
# Try it
species, confidence = predict("../data/raw/image.png")
print(f"Predicted: {species}  ({confidence:.1%} confidence)")

In [ ]:
for species, conf in predict_top3("../data/raw/image.png"):
    print(f"  {species:<15} {conf:.1%}")